
# 01/ Test DESC SN Ia metric SNNSNMMetrics  

try with `sne_nside = 16`

This metric is contributed by Philipe Gris. The metric creates a SN population at a range of redshifts, looks for detections pre and post peak, and then evaluates the number of SN that constitute a complete sample out to a redshift value defined by how well the color covariance can be defined. 

- author of corrections : Sylvie Dagoret-Campagne
- creation date : 2026-08-05
- copied and adapted (corrected) from https://github.com/lsst/rubin_sim_notebooks/tree/main/maf/science/testSNIa.ipynb

**`SNNSNMetric`**

`SNNSNMetric` (contributed by the DESC/Philippe Gris group) is a *cadence metric*: instead of just
counting visits, it asks a survey-strategy question — "given the actual sequence of LSST visits
(nights, filters, depths) at a given point on the sky, how well could we discover and characterize
Type Ia supernovae there?"

For each sky pixel it internally does the following, using ONLY the simulated visits that fall in that pixel:
1. Generate a grid of fake SNe Ia at different redshifts and explosion (peak) dates.
2. For each fake SN, check which of the real LSST visits would have observed it before/after peak
   brightness, and estimate the photometric uncertainty of each point (using a gamma/noise model).
3. Fit a simplified SALT2-like light curve and evaluate the color-uncertainty (sigma_color).
4. Find `zlim`: the highest redshift at which enough SNe pass quality cuts (sigma_color threshold,
   minimum epochs before/after peak) to be usable for cosmology.
5. Combine `zlim` with the SN Ia volumetric rate to get `nSN`: the expected number of well-measured
   SNe Ia out to that redshift, for that field/pixel and survey duration.


**How to read this notebook**

Unlike `00_SNIa_Fast.ipynb`, this notebook goes "under the hood" of `SNNSNMetric`:
- It first sets up the metric with explicit (non-default) parameters and explains what each controls.
- It then bypasses the usual `MetricBundleGroup.run_all()` loop to call `metric.run(...)` *directly*
  on the visits of a single, carefully chosen HEALPix pixel (one line of sight with a dense, long
  observing history), which is much faster than a full-sky run and lets you inspect intermediate data.
- It saves a few representative visit sequences to `test_simData.hdf` for reuse/regression testing.
- Finally it runs the metric over the *whole sky* twice: once on the WFD baseline survey, once on a
  Deep Drilling Field (DDF)-focused opsim run, so the WFD vs DDF cadences can be compared.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

import healpy as hp
import pandas as pd


import rubin_sim
from rubin_sim import maf
from rubin_sim.data import get_baseline

## 1. Configuration

In [ ]:
# RUBIN_SIM_DATA_DIR points to the local cache of rubin_sim/rubin_scheduler auxiliary data
# (opsim databases, dust maps, SN gamma/noise files, throughputs, ...).
# os.environ["RUBIN_SIM_DATA_DIR"] = "/users/dagoret/DATA/OpSim"
path_topdir = os.getenv("RUBIN_SIM_DATA_DIR")
print(f"path_topdir = {path_topdir}")

In [ ]:
# Baseline Survey
# get_baseline() returns the path to the reference LSST wide-fast-deep (WFD) cadence simulation,
# i.e. the list of simulated visits that will be fed to the metric below.
baseline_file = get_baseline()
# opsdb = maf.db.OpsimDatabase(baseline_file)
# opsdb =  maf.db.add_run_to_database(baseline_file)
run_name = os.path.split(baseline_file)[-1].replace(".db", "")

print(run_name)

In [ ]:
data_dir = None

if data_dir is None:
    import tempfile
    import os

    # Scratch directory for MAF's results database and any output products from this notebook.
    data_dir_itself = tempfile.TemporaryDirectory(prefix="01_maf_testSNIa_", dir=os.getcwd())
    data_dir = data_dir_itself.name

print(f"Using the {data_dir_itself.name} directory for output of this notebook")

In [ ]:
# Set up MAF output
# ResultsDb is MAF's bookkeeping database that tracks which metric bundles were run/plotted here.
out_dir = data_dir
resultsDb = maf.db.ResultsDb(out_dir=out_dir)

## 2 SNNSNMMetrics

### 2.0 Check info on SNNSNMMetrics 

In [ ]:
# to view the signature of the Metrics class
%pinfo maf.SNNSNMetric

In [ ]:
# to view the code of the metrics
# %psource maf.SNNSNMetric

### 2.1 Define the slicer, the metrics and  bundle

In [ ]:
#  Set up to time it at one point in the sky

# Clip the color scale of sky-map plots at the 95th percentile so outlier pixels don't dominate.
plot_dict = {"percentile_clip": 95.0, "n_ticks": 5}

# Finer HEALPix resolution than the fast notebook: nside=16 -> 12*16^2 = 3072 pixels over the sky.
sne_nside = 16
# DustMap provides the Milky Way (Schlegel/SFD-type) extinction E(B-V) at each slice point, used
# below to reject lines of sight with too much Galactic dust extinction (hard_dust_cut).
dustmap = maf.DustMap(nside=sne_nside)

# summary metrics
# Collapse the per-pixel skymap to single numbers: median/mean over the sky and a total count.
sn_summary = [maf.MedianMetric(), maf.MeanMetric(), maf.SumMetric(metric_name="Total detected")]

# Healpix slicer
# Partitions the sky into equal-area pixels; visits are dispatched per pixel to the metric.
slicer = maf.HealpixSlicer(nside=sne_nside, use_cache=False)

# SNNSNMetric metrics
# Parameters controlling the fake-SN population and the quality cuts applied to their light curves:
metric = maf.SNNSNMetric(
    n_bef=3,  # minimum number of well-measured epochs required BEFORE peak brightness
    n_aft=8,  # minimum number of well-measured epochs required AFTER peak brightness
    coadd_night=True,  # coadd multiple exposures taken the same night in the same filter
    add_dust=False,  # whether to apply the Milky Way dust extinction map to the fake SNe
    hard_dust_cut=0.25,  # reject lines of sight with E(B-V) above this value (too much extinction)
    zmin=0.1,  # lower edge of the simulated-SN redshift grid
    zmax=0.5,  # upper edge of the simulated-SN redshift grid
    z_step=0.03,  # redshift grid spacing used to scan for the completeness limit zlim
    daymax_step=3.0,  # spacing (days) of the grid of simulated SN peak (explosion) dates
    zlim_coeff=0.95,  # efficiency threshold (fraction of SNe passing quality cuts) defining zlim
    gamma_name="gamma_WFD.hdf5",  # photometric-noise model (LSST SNR "gamma" parameter) file to use
    verbose=False,
)

# Bundle
# Packages metric + slicer + SQL constraint (None = use all visits) + plotting/summary options.
bundle = maf.MetricBundle(
    metric,
    slicer,
    None,
    plot_dict=plot_dict,
    maps_list=[dustmap],
    summary_metrics=sn_summary,
    run_name=run_name,
)

### 2.2 Create the bundle group 

In [ ]:
# Bundle group
# Connects the bundle to the baseline opsim database and the output directory/results db.
bg = maf.MetricBundleGroup({"sn": bundle}, baseline_file, out_dir, resultsDb)

### 2.3 Examine/ Debug Bundle Group input

In [ ]:
# Inspect the signature/docstring of get_data (the method that queries the opsim database for visits).
%pinfo bg.get_data

### 2.4 Get the simulated data and set up the slicer so we can test ONE point

In [ ]:
# Behind the scenes stuff to get the simulated data and set up the slicer so we can test ONE point
# constraint
# set_current/get_data query all visits matching the (empty) SQL constraint from the opsim database.
bg.set_current("")
# Query the data from the database.
bg.get_data("")
# retreive data
# sim_data is the full numpy structured array of visits (MJD, filter, depth, sky position, ...).
sim_data = bg.sim_data
# Assign each visit to its HEALPix pixel so we can later grab "the visits seen by pixel X".
bundle.slicer.setup_slicer(sim_data)
# Add the dust extinction
# Attach the per-pixel Milky Way E(B-V) values computed by the dust map to the slicer's slice points.
slicer.slice_points = dustmap.run(slicer.slice_points)

### 2.5  Find a good spot on the sky with lots of visits

In [ ]:
# Find a good spot on the sky with lots of visits
# Loop over all HEALPix pixels and count how many visits ('idxs') fall in each, then pick the
# pixel with the MOST visits -- a good, densely-observed line of sight for a quick manual test.
len_visits = []
# loop on the slices
for s in bundle.slicer:
    len_visits.append(len(s["idxs"]))
lenvisits = np.array(len_visits)

idx = np.where(lenvisits == np.max(lenvisits))[0][0]
print(f"index with hightest count :\t idx = {idx}")
print("bundle.slicer[idx] :\t ", bundle.slicer[idx])

In [ ]:
# indexes sorted by decreasing order
idx_top10 = np.argsort(lenvisits)[-10:][::-1]

# Visit counts
top10_values = lenvisits[idx_top10]

print("slice indexes :\t", idx_top10)
print("pixel counts  :\t", top10_values)

In [ ]:
# sid = 890 - a point with a single usable season
# Use a fixed, previously identified pixel index instead (chosen for having useful cadence features).
sid = 2593
bundle.slicer[sid]

### 2.4 Run only only ONE  Pixel of the Healpix slicer
- the goal is ti estimate the time in One Slice
- It is possible to run the metrics on a single data slice

In [ ]:
%%time
# Call the metric directly on the visits belonging to a single pixel (bypassing run_all()).
# This is much faster than a full-sky run and returns the raw per-pixel result (n_sn, zlim, ...)
# for that one line of sight -- handy for debugging or timing the metric.
metric.run(sim_data[bundle.slicer[sid]["idxs"]], bundle.slicer[sid]["slice_point"])

In [ ]:
# Rough estimate of the total full-sky runtime by extrapolating the single-slice timing above
# (9 s/slice assumed here) to all pixels of the slicer.
print(
    len(slicer) * 9 / 60 / 60 / 2, "hrs best guess metric run time"
)  # /2 because maybe half sky doesn't have visits

### 2.5 Inspect the full structured array of simulated visits 

In [ ]:
# Inspect the full structured array of simulated visits (all columns: MJD, filter, depth, etc.).
sim_data

### 2.6 Create some metric test data

In [ ]:
# Take a moment and create some metric test data
# Build several representative visit subsets for pixel 2593, used below to populate a small
# regression-test HDF5 file (test_simData.hdf) covering different cadence/depth scenarios.

# extract the slice_data
s = sim_data[bundle.slicer[2593]["idxs"]]

# sort by MJD
s.sort(order="observationStartMJD")

# Find season gaps: consecutive nights more than 80 nights apart mark a new observing season.
dd = np.where(np.diff(s["night"]) > 80)[0] + 1

# Pick one full observing season (between two season-gap boundaries) as the reference sequence.
o = s[dd[3] : dd[4]]

fig, axs = plt.subplots(2, 1, figsize=(14, 5))
ax1, ax2 = axs.flatten()

n, b, p = ax1.hist(o["night"], bins=100)
# Same season but with DDF (Deep Drilling Field) visits removed -> a "WFD-only" cadence variant.
o2 = o[np.where(o["scheduler_note"] != "DD:ELAISS1")]
n, b, p = ax2.hist(o2["night"], bins=100)

plt.show()

In [ ]:
o["scheduler_note"]

In [ ]:
# A "shallower" variant: artificially degrade the 5-sigma depth by 4 mag to simulate worse conditions.
o3 = o2.copy()
o3["fiveSigmaDepth"] = o3["fiveSigmaDepth"] - 4
len(o), len(o2)
# A "single 20s exposure" variant instead of the usual multi-exposure visits.
o4 = o2.copy()
o4["numExposures"] = 1
o4["visitExposureTime"] = 20
# A "single exposure, default exposure time" variant.
o5 = o2.copy()
o5["numExposures"] = 1

In [ ]:
# Save all the visit-sequence variants built above into one HDF5 file (one key per scenario),
# for reuse as fixed test inputs in other notebooks/unit tests without re-querying the opsim db.
pd.DataFrame(sim_data[bundle.slicer[2593]["idxs"]]).to_hdf("test_simData.hdf", key="dense_pointing", mode="w")
pd.DataFrame(sim_data[bundle.slicer[890]["idxs"]]).to_hdf("test_simData.hdf", key="sparse_pointing", mode="a")
pd.DataFrame(o2).to_hdf("test_simData.hdf", key="one_season_noDD", mode="a")
pd.DataFrame(o).to_hdf("test_simData.hdf", key="one_season_wDD", mode="a")
pd.DataFrame(o3).to_hdf("test_simData.hdf", key="one_season_shallow", mode="a")
pd.DataFrame(o4).to_hdf("test_simData.hdf", key="one_season_singleExp_20", mode="a")
pd.DataFrame(o5).to_hdf("test_simData.hdf", key="one_season_singleExp_30", mode="a")

### 2.7 Run all slices

In [ ]:
%%time 
# -- this will run the whole sky, so could be close to the estimate above
# Now run the metric over EVERY HEALPix pixel (the normal MAF workflow), not just the one test pixel.
bg.run_all()

### 2.8 Plot metrics

In [ ]:
# Render the resulting sky maps (nSN, zlim, and any other reduced quantities).
bg.plot_all(closefigs=False)

### 2.9 Access to Metrics results

In [ ]:
# Raw per-pixel metric result for the first valid (unmasked) pixel.
bundle.metric_values.compressed()[0]

In [ ]:
# The 'reduce' values of the metric got stored in the bundle dict in the bungle group
# (which is why we usually set this as a dictionary outside of the metricBundleGroup call .. whoops.
# SNNSNMetric returns a compound value per pixel; MAF splits it via reduce_* methods into separate
# named bundles (e.g. n_sn, zlim), collected here in bg.bundle_dict.
bg.bundle_dict

In [ ]:
# The nSN and zlim values are pulled out in those reduce methods, into their own bundles.
# Expected number of well-measured SNe Ia per pixel (first 10 valid pixels).
bdict = bg.bundle_dict
bdict["SNNSNMetric_reducen_sn"].metric_values.compressed()[0:10]

In [ ]:
# Redshift completeness limit zlim per pixel (first 10 valid pixels).
bdict["SNNSNMetric_reducezlim"].metric_values.compressed()[0:10]

In [ ]:
# Print the sky-averaged summary values (median/mean/sum, as configured in sn_summary above)
# for each reduced quantity (n_sn, zlim).
for k in bdict:
    print(k, bdict[k].summary_values)
    display(pd.Series(bdict[k].summary_values))

In [ ]:
for k, val in bdict.items():
    display(pd.Series(val.summary_values, name=k))

## 3.0 Try another opsim run

### 3.1 Define the new run

In [ ]:
# run_name = 'rolling_bulge_ns2_rw0.9_v2.0_10yrs'
# Now repeat the whole exercise on a DIFFERENT opsim run: a DDF (Deep Drilling Field)-focused
# strategy, so its SN Ia yield/depth can be compared against the WFD baseline used above.
run_name = "ddf_sd_v5.3.0_10yrs"
opsdb = os.path.join(path_topdir, f"{run_name}.db")

### 3.2 Define the slicer and metrics and Bundle

In [ ]:
plot_dict = {"percentile_clip": 95.0, "n_ticks": 5}

sne_nside = 16
dustmap = maf.DustMap(nside=16)
sn_summary = [maf.MedianMetric(), maf.MeanMetric(), maf.SumMetric(metric_name="Total detected")]
slicer = maf.HealpixSlicer(nside=sne_nside, use_cache=False)
# Wider redshift range here (zmax=1.2 instead of 0.5) since DDF visits are deeper and can detect
# SNe Ia at higher redshift than the WFD baseline.
metric = maf.SNNSNMetric(
    n_bef=3,
    n_aft=8,
    coadd_night=True,
    add_dust=False,
    hard_dust_cut=0.25,
    zmin=0,
    zmax=1.2,
    z_step=0.03,
    daymax_step=3.0,
    zlim_coeff=0.95,
    gamma_name="gamma_WFD.hdf5",
    verbose=False,
)
bundle2 = maf.MetricBundle(
    metric,
    slicer,
    None,
    run_name=run_name,
    plot_dict=plot_dict,
    maps_list=[dustmap],
    summary_metrics=sn_summary,
)

bg = maf.MetricBundleGroup({"sn": bundle2}, opsdb, out_dir, resultsDb)

### 3.3 Run the metrics

In [ ]:
%%time
# Run the metric over all pixels of the DDF-focused opsim run.
bg.run_all()

### 3.4 Shwo metrics results

In [ ]:
# Print sky-averaged summary values for the DDF run, to compare against the WFD baseline above.
bdict = bg.bundle_dict.copy()
for k in bdict:
    print(k, bdict[k].summary_values)

In [ ]:
for k, val in bdict.items():
    display(pd.Series(val.summary_values, name=k))

### 3.5 Plot metrics results

In [ ]:
# Plot the DDF-run sky maps (nSN, zlim) for visual comparison with the WFD baseline maps above.
bg.plot_all(closefigs=False)